# NB04 해시 함수 — 지문을 찍는 일방향 함수

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB04-1: SHA-256 지문 찍기 — 한 글자 차이
import hashlib

def sha256_hex(s: str) -> str:
    """문자열을 UTF-8 바이트로 바꿔 SHA-256 다이제스트를 16진수 64자로 돌려준다."""
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

for s in ["hello", "hellp", "Hello", "hello "]:
    print(f"{s!r:10} -> {sha256_hex(s)}")

def differing_bits(h1: str, h2: str) -> int:
    """두 16진수 다이제스트에서 서로 다른 비트의 개수(해밍 거리)."""
    return bin(int(h1, 16) ^ int(h2, 16)).count("1")

print()
print("hello vs hellp : 다른 비트", differing_bits(sha256_hex("hello"), sha256_hex("hellp")), "/ 256")
print("hello vs Hello : 다른 비트", differing_bits(sha256_hex("hello"), sha256_hex("Hello")), "/ 256")
print("hello vs 'hello ': 다른 비트", differing_bits(sha256_hex("hello"), sha256_hex("hello ")), "/ 256")

'hello'    -> 2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
'hellp'    -> fdd7585e08c4e2afd71dcabdb4636c89d557a3f42db9e2040c8bbd1708aa4ce7
'Hello'    -> 185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969
'hello '   -> 5e3235a8346e5a4585f8c58562f5052b8fe26a3bb122e1e96c76784964dfc461

hello vs hellp : 다른 비트 131 / 256
hello vs Hello : 다른 비트 125 / 256
hello vs 'hello ': 다른 비트 139 / 256


In [ ]:
# NB04-2: 눈사태 효과 — 입력 비트를 하나씩 뒤집어 출력이 얼마나 바뀌는지 평균을 낸다
import hashlib

msg = b"The Times 03/Jan/2009 Chancellor on brink of second bailout for banks"
base = hashlib.sha256(msg).digest()

def flip_bit(data: bytes, i: int) -> bytes:
    """바이트열의 i번째 비트(0부터)를 뒤집는다."""
    b = bytearray(data)
    b[i // 8] ^= 1 << (i % 8)
    return bytes(b)

diffs = []
for i in range(len(msg) * 8):          # 입력의 모든 비트(69바이트 × 8 = 552개)를 한 번씩 뒤집는다
    h = hashlib.sha256(flip_bit(msg, i)).digest()
    diffs.append(sum(bin(a ^ b).count("1") for a, b in zip(base, h)))

print("뒤집은 입력 비트 수:", len(diffs))
print("출력에서 바뀐 비트 수 — 최소 %d, 최대 %d, 평균 %.1f (256비트 중)" % (min(diffs), max(diffs), sum(diffs) / len(diffs)))

뒤집은 입력 비트 수: 552
출력에서 바뀐 비트 수 — 최소 101, 최대 150, 평균 127.5 (256비트 중)


In [ ]:
# NB04-3: 검사합 위조는 즉시, 해시 위조는 전수 탐색
import hashlib

def checksum8(data: bytes) -> int:
    """바이트 합의 끝자리(mod 256) — 우연한 오류용 검사합."""
    return sum(data) % 256

original = b"Pay to Alice 100"
target = checksum8(original)
forged = bytearray(b"Pay to Bob   100 ")         # 수취인을 바꾸고
forged.append((target - checksum8(forged)) % 256)   # 뒤에 바이트 하나를 붙여 끝자리를 맞춘다
print("원본 검사합  :", target)
print("위조문 검사합:", checksum8(forged), "| 위조문:", bytes(forged))

def toy_hash16(data: bytes) -> int:
    """SHA-256의 앞 16비트만 쓰는 장난감 해시(실사용 금지 — 설명용)."""
    return int.from_bytes(hashlib.sha256(data).digest()[:2], "big")

t = toy_hash16(original)
attempts = 0
while True:
    attempts += 1
    candidate = b"Pay to Bob   100 #" + str(attempts).encode()
    if toy_hash16(candidate) == t:
        break
print("16비트 장난감 해시를 맞추기까지 시도 횟수:", attempts, "(기대값 2^16 = 65,536)")
print("256비트 전체를 맞추려면 기대 시도 횟수는 2^256 ≈ %.1e" % 2**256)

원본 검사합  : 220
위조문 검사합: 220 | 위조문: b'Pay to Bob   100 k'
16비트 장난감 해시를 맞추기까지 시도 횟수: 27882 (기대값 2^16 = 65,536)
256비트 전체를 맞추려면 기대 시도 횟수는 2^256 ≈ 1.2e+77


In [ ]:
# NB04-4: 생일 역설 — 16비트 장난감 해시에서 첫 충돌까지 뽑은 개수
import hashlib, math

def toy_hash16(data: bytes) -> int:
    return int.from_bytes(hashlib.sha256(data).digest()[:2], "big")

def draws_until_collision(run: int) -> int:
    """run번째 실험: 서로 다른 입력을 하나씩 넣어 처음 같은 해시가 나올 때까지의 개수."""
    seen = {}
    k = 0
    while True:
        k += 1
        h = toy_hash16(f"run{run}-item{k}".encode())
        if h in seen:
            return k
        seen[h] = k

results = [draws_until_collision(r) for r in range(200)]
N = 2**16
print("실험 횟수 200, 출력 공간 N = %d" % N)
print("첫 충돌까지 개수 — 평균 %.1f, 최소 %d, 최대 %d" % (sum(results) / len(results), min(results), max(results)))
print("이론 기대값 sqrt(pi*N/2) = %.1f" % math.sqrt(math.pi * N / 2))
print("충돌 확률이 1/2이 되는 n ≈ 1.18*sqrt(N) = %.0f" % (1.18 * math.sqrt(N)))
print("SHA-256(N = 2^256)이면 1.18*sqrt(N) ≈ %.2e" % (1.18 * math.sqrt(2**256)))

실험 횟수 200, 출력 공간 N = 65536
첫 충돌까지 개수 — 평균 325.1, 최소 19, 최대 754
이론 기대값 sqrt(pi*N/2) = 320.8
충돌 확률이 1/2이 되는 n ≈ 1.18*sqrt(N) = 302
SHA-256(N = 2^256)이면 1.18*sqrt(N) ≈ 4.02e+38


In [ ]:
# NB04-5: SHA-256을 손으로 — 표준 명세의 반복 구조를 눈으로 본다 (공부용, 실사용 금지)
import hashlib, struct

def icbrt(n: int) -> int:
    """정수 세제곱근(내림) — 부동소수 오차 없이 상수를 만들기 위해 정수 연산만 쓴다."""
    x = 1 << ((n.bit_length() + 2) // 3)
    while True:
        y = (2 * x + n // (x * x)) // 3
        if y >= x:
            return x
        x = y

PRIMES = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,83,89,97,101,103,107,109,113,
          127,131,137,139,149,151,157,163,167,173,179,181,191,193,197,199,211,223,227,229,233,239,241,251,
          257,263,269,271,277,281,283,293,307,311]
# 상수 K: 처음 64개 소수의 세제곱근 소수부 앞 32비트 (FIPS 180-4 §4.2.2) — cbrt(p)·2^32 의 정수부를 2^32 로 나눈 나머지
K = [icbrt(p << 96) & 0xFFFFFFFF for p in PRIMES]
# 초기 해시값 H0: 처음 8개 소수의 제곱근 소수부 앞 32비트 (FIPS 180-4 §5.3.3)
import math
H0 = [math.isqrt(p << 64) & 0xFFFFFFFF for p in PRIMES[:8]]

def rotr(x, n): return ((x >> n) | (x << (32 - n))) & 0xFFFFFFFF

def compress(H, block):
    """압축 함수: 256비트 상태 H와 512비트 블록 하나 → 새 상태 (FIPS 180-4 §6.2.2)"""
    w = list(struct.unpack(">16I", block))
    for t in range(16, 64):
        s0 = rotr(w[t-15], 7) ^ rotr(w[t-15], 18) ^ (w[t-15] >> 3)
        s1 = rotr(w[t-2], 17) ^ rotr(w[t-2], 19) ^ (w[t-2] >> 10)
        w.append((w[t-16] + s0 + w[t-7] + s1) & 0xFFFFFFFF)
    a, b, c, d, e, f, g, h = H
    for t in range(64):
        S1 = rotr(e, 6) ^ rotr(e, 11) ^ rotr(e, 25)
        ch = (e & f) ^ (~e & g)
        T1 = (h + S1 + ch + K[t] + w[t]) & 0xFFFFFFFF
        S0 = rotr(a, 2) ^ rotr(a, 13) ^ rotr(a, 22)
        maj = (a & b) ^ (a & c) ^ (b & c)
        T2 = (S0 + maj) & 0xFFFFFFFF
        h, g, f, e, d, c, b, a = g, f, e, (d + T1) & 0xFFFFFFFF, c, b, a, (T1 + T2) & 0xFFFFFFFF
    return [(x + y) & 0xFFFFFFFF for x, y in zip(H, [a, b, c, d, e, f, g, h])]

def sha256_manual(msg: bytes, trace=False) -> str:
    L = len(msg) * 8
    padded = msg + b"\x80" + b"\x00" * ((55 - len(msg)) % 64) + struct.pack(">Q", L)   # §5.1.1 채움
    H = H0[:]
    for i in range(0, len(padded), 64):
        H = compress(H, padded[i:i+64])
        if trace:
            print(f"  블록 {i//64 + 1} 처리 후 상태 H^({i//64 + 1}) = {''.join(f'{x:08x}' for x in H)}")
    return "".join(f"{x:08x}" for x in H)

for m in [b"", b"abc", b"hello"]:
    ok = sha256_manual(m) == hashlib.sha256(m).hexdigest()
    print(f"{m!r:9} 손 구현 == hashlib ? {ok}")

two_blocks = b"a" * 100          # 100바이트 → 채움 뒤 512비트 블록 2개
print("100바이트 메시지(블록 2개)의 상태 변화:")
digest = sha256_manual(two_blocks, trace=True)
print("  최종 다이제스트            =", digest)
print("  hashlib                    =", hashlib.sha256(two_blocks).hexdigest())

b''       손 구현 == hashlib ? True
b'abc'    손 구현 == hashlib ? True
b'hello'  손 구현 == hashlib ? True
100바이트 메시지(블록 2개)의 상태 변화:
  블록 1 처리 후 상태 H^(1) = df5bb81ce81e0626fb45a8944fd40f31b25e6816d6d499c1ab90492900635e66
  블록 2 처리 후 상태 H^(2) = 2816597888e4a0d3a36b82b83316ab32680eb8f00f8cd3b904d681246d285a0e
  최종 다이제스트            = 2816597888e4a0d3a36b82b83316ab32680eb8f00f8cd3b904d681246d285a0e
  hashlib                    = 2816597888e4a0d3a36b82b83316ab32680eb8f00f8cd3b904d681246d285a0e
